# Session 8: Data Aggregation & Joins

**Course:** Python for Data Engineering  
**Phase 2:** Data Handling & Transformation

**What we'll cover:**
- GroupBy operations
- Aggregation functions
- Joins (merge) — inner, left, right, outer
- Combining DataFrames with concat

**Data files:** `data/orders.csv`, `data/customers.csv`, `data/departments.csv`, `data/employees.csv`

**Note:** This session is demo-heavy. Follow along by running each cell. The lab at the end ties everything together.

In [ ]:
import pandas as pd
import numpy as np

---

## 1. GroupBy — Split, Apply, Combine

GroupBy is how you answer questions like:
- Total revenue **per region**
- Average salary **per department**
- Number of orders **per customer**

It works in three steps:
1. **Split** — group rows by a column value
2. **Apply** — run an aggregation (sum, mean, count, etc.)
3. **Combine** — return the results as a new DataFrame

In [ ]:
# Load orders

orders = pd.read_csv("data/orders.csv")
orders["total"] = orders["quantity"] * orders["unit_price"]
orders

In [ ]:
# Basic groupby — total revenue per region

revenue_by_region = orders.groupby("region")["total"].sum()
print("Revenue by region:")
print(revenue_by_region)

In [ ]:
# Count orders per customer

orders_per_customer = orders.groupby("customer")["order_id"].count()
print("Orders per customer:")
print(orders_per_customer)

In [ ]:
# Multiple aggregations at once

customer_summary = orders.groupby("customer").agg(
    total_orders=("order_id", "count"),
    total_spent=("total", "sum"),
    avg_order_value=("total", "mean"),
    total_items=("quantity", "sum"),
)
print("Customer summary:")
print(customer_summary)

In [ ]:
# Group by multiple columns

region_product = orders.groupby(["region", "product"]).agg(
    total_qty=("quantity", "sum"),
    revenue=("total", "sum"),
).reset_index()

print("Revenue by region and product:")
print(region_product)

In [ ]:
# Sort grouped results — top products by revenue

product_revenue = orders.groupby("product")["total"].sum().sort_values(ascending=False)
print("Products by revenue (highest first):")
print(product_revenue)

### GroupBy aggregation reference

| Function | What it does |
|----------|-------------|
| `.sum()` | Total |
| `.mean()` | Average |
| `.count()` | Number of rows |
| `.min()` / `.max()` | Min / Max |
| `.median()` | Median |
| `.std()` | Standard deviation |
| `.first()` / `.last()` | First / Last value |
| `.nunique()` | Count of unique values |
| `.agg(...)` | Multiple custom aggregations |

---

## 2. Pivot Tables

A pivot table is a groupby reshaped as a cross-tab — rows become one dimension, columns become another. Common for reports.

In [ ]:
# Pivot table — revenue by region (rows) and product (columns)

pivot = orders.pivot_table(
    values="total",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0,
)
print("Revenue pivot:")
print(pivot)

In [ ]:
# Add row and column totals with margins

pivot_totals = orders.pivot_table(
    values="total",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
)
print("With totals:")
print(pivot_totals)

---

## 3. Joins (Merging DataFrames)

In databases, you join tables. In pandas, you use `pd.merge()` or `df.merge()`. Same idea — combine two DataFrames based on a shared column.

### Join types

| Join | Keeps rows from |
|------|----------------|
| `inner` | Only rows that match in **both** DataFrames |
| `left` | All rows from **left** DataFrame, matched rows from right |
| `right` | All rows from **right** DataFrame, matched rows from left |
| `outer` | All rows from **both** (NaN where no match) |

In [ ]:
# Load the datasets we'll join

customers = pd.read_csv("data/customers.csv")
print("Customers:")
print(customers)

print("\nOrders (first 5):")
print(orders.head())

In [ ]:
# Inner join — only customers who have orders
# match on customer name

inner = pd.merge(
    orders, customers,
    left_on="customer",
    right_on="name",
    how="inner",
)
print(f"Inner join: {len(inner)} rows")
print(inner[["order_id", "customer", "product", "city", "email"]].head())

In [ ]:
# Left join — all orders, with customer info where available
# orders without a matching customer will have NaN for city/email

left = pd.merge(
    orders, customers,
    left_on="customer",
    right_on="name",
    how="left",
)
print(f"Left join: {len(left)} rows")
print(left[["order_id", "customer", "product", "city"]].head())

# Check for unmatched orders
unmatched = left[left["city"].isna()]
if len(unmatched) > 0:
    print(f"\nOrders without customer info: {len(unmatched)}")
    print(unmatched[["order_id", "customer"]])
else:
    print("\nAll orders matched to customers.")

In [ ]:
# Right join — all customers, with their orders if any
# customers without orders will have NaN for order fields

right = pd.merge(
    orders, customers,
    left_on="customer",
    right_on="name",
    how="right",
)
print(f"Right join: {len(right)} rows")

# Eve Williams has no orders — she should show up with NaN
no_orders = right[right["order_id"].isna()]
print(f"\nCustomers with no orders: {len(no_orders)}")
print(no_orders[["name", "city", "order_id"]])

In [ ]:
# Outer join — everything from both sides

outer = pd.merge(
    orders, customers,
    left_on="customer",
    right_on="name",
    how="outer",
)
print(f"Outer join: {len(outer)} rows")
print(f"Nulls per column:")
print(outer.isnull().sum())

### When to use which join

| Scenario | Join type |
|----------|----------|
| Show only records that exist in both tables | `inner` |
| Keep all orders, enrich with customer data | `left` |
| Find customers who never ordered | `right` or `left` (swap order) |
| Full picture — everything from both sides | `outer` |

**In data pipelines:** `left` join is by far the most common — you have a fact table (orders) and you enrich it with dimension tables (customers, products, etc.).

---

## 4. Merging on Same Column Name

When both DataFrames share a column name, you can use `on=` instead of `left_on` / `right_on`.

In [ ]:
# Load employees and departments

employees = pd.read_csv("data/employees.csv")
employees["salary"] = employees["salary"].str.replace("$", "", regex=False)
employees["salary"] = employees["salary"].str.replace(",", "", regex=False)
employees["salary"] = pd.to_numeric(employees["salary"], errors="coerce")
employees["name"] = employees["name"].str.strip().str.title()
employees["department"] = employees["department"].str.strip().str.lower()

departments = pd.read_csv("data/departments.csv")

print("Employees:")
print(employees)
print("\nDepartments:")
print(departments)

In [ ]:
# Merge employees with department info
# employees.department links to departments.dept_code

emp_dept = pd.merge(
    employees, departments,
    left_on="department",
    right_on="dept_code",
    how="left",
)
print("Employees with department details:")
print(emp_dept[["name", "department", "dept_name", "location", "salary"]])

---

## 5. Concatenating DataFrames

Merging combines DataFrames **sideways** (adding columns). Concatenation stacks them **vertically** (adding rows). Use `pd.concat()` when you have the same columns.

In [ ]:
# Simulate data arriving in batches — like daily exports

batch_jan = pd.DataFrame({
    "date": ["2024-01-15", "2024-01-16"],
    "product": ["Laptop", "Mouse"],
    "revenue": [1999.98, 299.90],
})

batch_feb = pd.DataFrame({
    "date": ["2024-02-10", "2024-02-12"],
    "product": ["Keyboard", "Monitor"],
    "revenue": [397.50, 349.99],
})

# Stack them vertically
all_data = pd.concat([batch_jan, batch_feb], ignore_index=True)
print("Combined batches:")
print(all_data)

In [ ]:
# Concat with mismatched columns — NaN fills the gaps

batch_mar = pd.DataFrame({
    "date": ["2024-03-01"],
    "product": ["Headphones"],
    "revenue": [179.97],
    "region": ["north"],  # extra column
})

combined = pd.concat([all_data, batch_mar], ignore_index=True)
print("With mismatched columns:")
print(combined)

---

## 6. Common Patterns — GroupBy + Join

In real pipelines, you often aggregate first, then join the results. Here's a realistic example.

In [ ]:
# Build a customer report: join orders with customers, then aggregate

# Step 1: Join orders with customer info
enriched = pd.merge(
    orders, customers,
    left_on="customer",
    right_on="name",
    how="left",
)

# Step 2: Aggregate per customer
report = enriched.groupby(["customer", "city"]).agg(
    total_orders=("order_id", "count"),
    total_revenue=("total", "sum"),
    avg_order=("total", "mean"),
    products_bought=("product", "nunique"),
).reset_index()

# Step 3: Sort by revenue
report = report.sort_values("total_revenue", ascending=False)

print("Customer report:")
print(report)

In [ ]:
# Another pattern: find top product per region

region_product_rev = orders.groupby(["region", "product"])["total"].sum().reset_index()

# Get the top product per region using idx of max
top_per_region = region_product_rev.loc[
    region_product_rev.groupby("region")["total"].idxmax()
]

print("Top product per region by revenue:")
print(top_per_region)

---

## Lab: Build an Employee Department Report

Combine employees and departments data to build a department-level summary report.

**Steps:**
1. Load `data/employees.csv` — clean names (strip, title), clean salary (remove $, to numeric), clean department (strip, lower)
2. Load `data/departments.csv`
3. **Join** employees with departments (employees.department → departments.dept_code), use a left join
4. **Aggregate** by department:
   - Number of employees
   - Total salary cost
   - Average salary
   - Average age
5. **Join** the budget from departments to see: department name, headcount, total salary, budget, and remaining budget (budget - total salary)
6. Sort by total salary descending
7. Print the final report

In [ ]:
import pandas as pd

# Your code here

---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| GroupBy | `df.groupby(col)[val].agg()` — split-apply-combine |
| Multi-agg | `.agg(name=(col, func))` — multiple named aggregations |
| Pivot tables | `pivot_table()` — cross-tab style aggregation |
| Inner join | `how='inner'` — only matching rows |
| Left join | `how='left'` — keep all left rows, match from right |
| Right join | `how='right'` — keep all right rows |
| Outer join | `how='outer'` — everything from both sides |
| Concat | `pd.concat()` — stack DataFrames vertically |

**Key patterns:**
- GroupBy + sort = ranked reports (top customers, top products)
- Left join = enrich fact tables with dimension data
- Concat = combine data from multiple sources or time periods
- Always check for NaN after joins — unmatched rows show up as nulls

**Next session:** Working with Dates & Time — datetime operations, time-based transformations, and time series basics.